# Notebook 07 — Features de couchage vs boiterie (Winter 2019)

**Hypothèse testée :** la boiterie légère ne se voit pas dans le *nombre de pas* (montré au notebook 06),
mais peut-être dans le *comportement de couchage* — un marqueur de boiterie reconnu dans la littérature
(les vaches boiteuses modifient souvent leurs durées et fréquences de couchage).

**C'est la dernière tentative honnête d'extraire un signal de boiterie des données IceTag existantes.**

**Features de couchage testées :**
- heures couché / jour
- nombre de couchers / jour (lying-down events)
- durée moyenne d'une période couchée
- agitation (transitions par heure couchée)
- fragmentation (couchers par heure couchée)
- ratio couchage jour / nuit
- temps debout / jour

**Données :** Winter 2019 (labels SLS synchrones, mars 2019), input 15-min déjà validé par le pipeline.

**Ce notebook ne modifie pas le mémoire.** Gestion d'attentes : le signal peut très bien rester absent
(boiterie trop légère) — auquel cas le résultat négatif sera documenté proprement.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, spearmanr
import warnings
warnings.filterwarnings('ignore')

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = PROJECT / 'Données completes' / 'Données accelerometres'
REPORTS = PROJECT / 'reports' / 'objective1_pipeline_icetag'
OUT = REPORTS / 'features_couchage'
OUT.mkdir(parents=True, exist_ok=True)

ICETAG_INPUT = REPORTS / 'winter_2019_pipeline_input_15min.csv'
SLS_FILE = (DATA_ROOT / 'Winter 2019' / 'Icetag' / 'IceTags_Data' /
            'IceTags-issues and reports' / 'Exercise Study - SLS Scores.xlsx')
print('OK' if ICETAG_INPUT.exists() and SLS_FILE.exists() else 'FICHIER MANQUANT')

OK


## 1. Scores cliniques SLS (référence)

In [2]:
sls = pd.read_excel(SLS_FILE).rename(columns={'Unnamed: 6': 'SLS_total'})
sls['Cow'] = sls['Cow'].astype(str)
sls_agg = sls.groupby('Cow')['SLS_total'].max().reset_index()
sls_agg['boiteuse'] = (sls_agg['SLS_total'] >= 2).astype(int)
print(f"Vaches scorées : {len(sls_agg)} | boiteuses (SLS>=2) : {sls_agg['boiteuse'].sum()}")

Vaches scorées : 33 | boiteuses (SLS>=2) : 13


## 2. Ingénierie des features de couchage (par vache)

À partir des bins de 15 min : `Lying Time` (durée couché), `Standing Time`, `Transitions Down`
(événements de coucher), `Transitions` (agitation). On agrège au jour puis on moyenne par vache.
Le jour = 06h-18h, la nuit = 18h-06h.

In [3]:
inp = pd.read_csv(ICETAG_INPUT)
inp['Cow'] = inp['Cow'].astype(str)
inp['Start'] = pd.to_datetime(inp['Start'])
inp['date'] = inp['Start'].dt.date
inp['hour'] = inp['Start'].dt.hour
inp['is_night'] = ((inp['hour'] >= 18) | (inp['hour'] < 6))

def hms_to_h(t):
    try:
        h, m, s = str(t).split(':')
        return (int(h) * 3600 + int(m) * 60 + float(s)) / 3600
    except Exception:
        return np.nan

inp['lying_h'] = inp['Lying Time'].apply(hms_to_h)
inp['standing_h'] = inp['Standing Time'].apply(hms_to_h)

# Agrégat journalier
daily = inp.groupby(['Cow', 'date']).agg(
    lying_h=('lying_h', 'sum'),
    standing_h=('standing_h', 'sum'),
    lying_events=('Transitions Down', 'sum'),
    transitions=('Transitions', 'sum'),
    lying_h_night=('lying_h', lambda x: x[inp.loc[x.index, 'is_night']].sum()),
    lying_h_day=('lying_h', lambda x: x[~inp.loc[x.index, 'is_night']].sum()),
).reset_index()

# Features dérivées au niveau jour
daily['mean_bout_min'] = (daily['lying_h'] * 60) / daily['lying_events'].replace(0, np.nan)
daily['restlessness'] = daily['transitions'] / daily['lying_h'].replace(0, np.nan)
daily['fragmentation'] = daily['lying_events'] / daily['lying_h'].replace(0, np.nan)
daily['day_night_ratio'] = daily['lying_h_day'] / daily['lying_h_night'].replace(0, np.nan)

# Moyenne par vache
feats = daily.groupby('Cow').agg(
    lying_h_per_day=('lying_h', 'mean'),
    standing_h_per_day=('standing_h', 'mean'),
    lying_events_per_day=('lying_events', 'mean'),
    mean_bout_min=('mean_bout_min', 'mean'),
    restlessness=('restlessness', 'mean'),
    fragmentation=('fragmentation', 'mean'),
    day_night_ratio=('day_night_ratio', 'mean'),
).reset_index()
print(f"Features de couchage calculées pour {len(feats)} vaches")
feats.round(2)

Features de couchage calculées pour 17 vaches


,Cow,lying_h_per_day,standing_h_per_day,lying_events_per_day,mean_bout_min,restlessness,fragmentation,day_night_ratio
0,2047,16.53,9.67,158.45,16.65,18.05,9.71,0.94
1,2056,12.81,12.77,64.51,19.35,8.37,4.94,0.67
2,2063,16.13,9.46,126.66,17.20,14.47,7.85,0.90
3,2069,15.12,12.30,65.25,26.36,6.78,4.03,0.89
4,2081,15.01,10.59,43.95,27.53,4.97,3.10,0.74
5,2083,13.15,7.73,114.62,7.19,15.42,8.51,0.88
6,3437,14.62,10.80,41.93,25.03,4.51,2.90,0.72
7,3443,16.87,8.57,97.14,23.38,10.23,5.69,0.81
8,5221,14.13,11.46,32.48,39.20,3.59,2.37,0.78
9,5246,11.71,14.04,61.69,19.78,9.42,5.52,0.73


## 3. Test de séparabilité univarié (couchage vs SLS)

In [4]:
m = feats.merge(sls_agg, on='Cow', how='inner')
print(f"Vaches comparables : {len(m)} | boiteuses : {m['boiteuse'].sum()} | saines/légères : {(m['boiteuse']==0).sum()}\n")

feature_cols = ['lying_h_per_day', 'standing_h_per_day', 'lying_events_per_day',
                'mean_bout_min', 'restlessness', 'fragmentation', 'day_night_ratio']
labels = {
    'lying_h_per_day': 'Heures couché / jour',
    'standing_h_per_day': 'Heures debout / jour',
    'lying_events_per_day': 'Nb couchers / jour',
    'mean_bout_min': 'Durée moy. coucher (min)',
    'restlessness': 'Agitation (trans/h couché)',
    'fragmentation': 'Fragmentation (couchers/h)',
    'day_night_ratio': 'Ratio couchage jour/nuit',
}
rows = []
for col in feature_cols:
    lame = m[m['boiteuse'] == 1][col].dropna()
    sain = m[m['boiteuse'] == 0][col].dropna()
    stat, p = mannwhitneyu(lame, sain, alternative='two-sided')
    rho, pr = spearmanr(m['SLS_total'], m[col])
    rows.append({'feature': labels[col], 'moy_boiteuses': round(lame.mean(), 2),
                 'moy_saines': round(sain.mean(), 2), 'p_value': round(p, 4),
                 'spearman_rho': round(rho, 3), 'signif_5pct': 'OUI' if p < 0.05 else 'non'})
result = pd.DataFrame(rows)
result.to_csv(OUT / 'separabilite_couchage.csv', index=False)
print(result.to_string(index=False))
print(f"\nFeatures significatives (p<0.05) : {(result['p_value'] < 0.05).sum()} / {len(result)}")

Vaches comparables : 16 | boiteuses : 5 | saines/légères : 11

                   feature  moy_boiteuses  moy_saines  p_value  spearman_rho signif_5pct
      Heures couché / jour          14.87       14.54   0.5833         0.000         non
      Heures debout / jour          11.21       10.76   0.9130         0.195         non
        Nb couchers / jour          83.00       63.46   0.2674         0.249         non
  Durée moy. coucher (min)          25.07       24.74   0.6612        -0.139         non
Agitation (trans/h couché)           9.61        7.01   0.2674         0.284         non
Fragmentation (couchers/h)           5.46        4.18   0.2674         0.284         non
  Ratio couchage jour/nuit           0.83        0.80   0.7427        -0.014         non

Features significatives (p<0.05) : 0 / 7


## 4. Plafond multivarié (combinaison de features de couchage)

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

X = m[feature_cols].fillna(m[feature_cols].median())
y = m['boiteuse'].values
clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
proba = cross_val_predict(clf, X, y, cv=LeaveOneOut(), method='predict_proba')[:, 1]
auc = roc_auc_score(y, proba)
bacc = balanced_accuracy_score(y, (proba >= 0.5).astype(int))
print(f"AUC (Leave-One-Out) : {auc:.3f}   (0.5 = hasard)")
print(f"Balanced accuracy   : {bacc:.3f}")
print()
if auc >= 0.75:
    print("=> SIGNAL EXPLOITABLE dans le comportement de couchage ! Piste à approfondir.")
elif auc >= 0.65:
    print("=> Signal faible mais présent. Le couchage fait un peu mieux que les pas.")
else:
    print("=> Pas de signal exploitable, même dans le couchage. La boiterie légère reste indétectable.")

# Comparaison avec le résultat des pas (notebook 06)
print("\nRappel notebook 06 (features d'activité/pas) : AUC = 0.24")
print(f"Ici (features de couchage)                   : AUC = {auc:.3f}")

AUC (Leave-One-Out) : 0.182   (0.5 = hasard)
Balanced accuracy   : 0.364

=> Pas de signal exploitable, même dans le couchage. La boiterie légère reste indétectable.

Rappel notebook 06 (features d'activité/pas) : AUC = 0.24
Ici (features de couchage)                   : AUC = 0.182


## 5. Visualisation

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(feature_cols), figsize=(3.4 * len(feature_cols), 4))
for ax, col in zip(axes, feature_cols):
    data = [m[m['boiteuse'] == 0][col].dropna(), m[m['boiteuse'] == 1][col].dropna()]
    ax.boxplot(data, labels=['Saines', 'Boit.'])
    for i, d in enumerate(data, start=1):
        ax.scatter(np.random.normal(i, 0.05, len(d)), d, alpha=0.6, s=25)
    ax.set_title(labels[col], fontsize=9)
fig.suptitle('Winter 2019 — features de couchage par statut clinique SLS', fontsize=12)
fig.tight_layout()
fig.savefig(OUT / 'boxplots_couchage.png', dpi=120, bbox_inches='tight')
print('Figure sauvegardée :', OUT / 'boxplots_couchage.png')
plt.show()

Figure sauvegardée : /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/features_couchage/boxplots_couchage.png


## 6. Conclusion

In [7]:
n_sig = int((result['p_value'] < 0.05).sum())
lines = []
lines.append('# Features de couchage vs boiterie — Winter 2019\n')
lines.append(f'Vaches : {len(m)} (boiteuses : {int(m["boiteuse"].sum())}, saines/légères : {int((m["boiteuse"]==0).sum())})\n')
lines.append('## Séparabilité univariée (couchage)')
lines.append(f'Features significatives (p<0.05) : {n_sig} / {len(result)}\n')
lines.append(result.to_string(index=False))
lines.append(f'\n## Plafond multivarié')
lines.append(f'AUC features de couchage = {auc:.3f} | features de pas (nb 06) = 0.24 | hasard = 0.5\n')
lines.append('## Conclusion')
if n_sig == 0 and auc < 0.65:
    lines.append(
        "Le comportement de couchage ne sépare pas non plus les vaches boiteuses des saines. "
        "Le signal de boiterie légère est absent de TOUTES les variables IceTag disponibles "
        "(activité ET couchage). \n\n"
        "=> Confirmation finale : la limite n'est pas l'algorithme ni les features choisies, "
        "mais le CAPTEUR. L'IceTag mesure la quantité de mouvement, pas l'asymétrie de démarche "
        "(le vrai marqueur de boiterie légère). Recommandation : capteurs de symétrie de démarche "
        "ou analyse vidéo pour les cas légers.")
else:
    lines.append(
        "Le comportement de couchage apporte un signal supérieur aux pas. Piste exploitable : "
        "intégrer ces features de couchage dans le pipeline et ré-évaluer contre le SLS.")
note = '\n'.join(lines)
(OUT / 'conclusion_couchage.md').write_text(note, encoding='utf-8')
print(note)

# Features de couchage vs boiterie — Winter 2019

Vaches : 16 (boiteuses : 5, saines/légères : 11)

## Séparabilité univariée (couchage)
Features significatives (p<0.05) : 0 / 7

                   feature  moy_boiteuses  moy_saines  p_value  spearman_rho signif_5pct
      Heures couché / jour          14.87       14.54   0.5833         0.000         non
      Heures debout / jour          11.21       10.76   0.9130         0.195         non
        Nb couchers / jour          83.00       63.46   0.2674         0.249         non
  Durée moy. coucher (min)          25.07       24.74   0.6612        -0.139         non
Agitation (trans/h couché)           9.61        7.01   0.2674         0.284         non
Fragmentation (couchers/h)           5.46        4.18   0.2674         0.284         non
  Ratio couchage jour/nuit           0.83        0.80   0.7427        -0.014         non

## Plafond multivarié
AUC features de couchage = 0.182 | features de pas (nb 06) = 0.24 | hasard = 0.5

## C